# 03 Follow-ups

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ['GE_DRIVE_ROOT'] = '/content/drive/MyDrive/hypothesaes_goemotions'
REPO_URL = 'https://github.com/vankadaruvijayberk/HypotheSAEs.git'
REPO_DIR = '/content/HypotheSAEs'

!rm -rf {REPO_DIR}   # always take the pushed version

## Install

In [ ]:
import os, subprocess
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)

!pip install -q -e {REPO_DIR}
!pip install -q -U datasets sentence-transformers transformers scikit-learn statsmodels "numpy<2.0.0"
!pip install -q -U vllm torchaudio torchvision nvidia-cuda-runtime-cu13
!pip uninstall -y -q torchcodec

import sys, importlib
for p in (REPO_DIR, os.path.join(REPO_DIR, 'goemotions')):
    if p not in sys.path:
        sys.path.insert(0, p)
importlib.invalidate_caches()

## Imports

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, spearmanr
import ge_pipeline as ge
from hypothesaes.embedding import get_local_embeddings
from hypothesaes.annotate import annotate_texts_with_concepts
ge.set_seed()
ge.redirect_annotation_cache()
ge.fix_cuda_libs()
print('free VRAM (GB):', ge.gpu_free_gb())

## Load Heldout and Hypotheses

In [ ]:
ds, NAMES = ge.load_goemotions()
test_texts = list(ds['test']['text'])
test_targets = ge.build_targets(ds['test']['labels'], NAMES)

interp = ge.read_json('06_interpretations')
evaluation = ge.read_json('07_evaluation')
hidx = evaluation['heldout_indices']            # same rows notebook 02 annotated
held_texts = [test_texts[i] for i in hidx]
held_targets = {t: test_targets[t][hidx] for t in test_targets}

selected = {t: [int(n) for n in v] for t, v in interp['selected'].items()}
neuron2hyp = {int(k): v['interpretation'] for k, v in interp['best'].items() if v['interpretation']}
union_hyps = sorted(set(neuron2hyp.values()))
print('heldout:', len(held_texts), '| hypotheses:', len(union_hyps))

## Cached Annotations

In [ ]:
missing = ge.annotation_cache_missing('goemotions_heldout', union_hyps, held_texts)
assert missing == 0, f'{missing} heldout annotations missing; rerun notebook 02'

llm = annotate_texts_with_concepts(texts=held_texts, concepts=union_hyps, model='Qwen/Qwen3-1.7B',
                                   cache_name='goemotions_heldout', n_workers=1,
                                   use_cache_only=True, uncached_value=0, max_words_per_example=64)
prev = pd.Series({h: float(llm[h].mean()) for h in union_hyps})
print('hypotheses with zero variance on heldout:', int((prev == 0).sum() + (prev == 1).sum()))
prev.describe()

## Rescore Main Evaluation

In [ ]:
# score_hypotheses fits a Logit on the annotation matrix; a hypothesis the annotator
# never fires on is constant, collinear with the intercept -> singular matrix, inf, NaN R2.
stable, dropped_all = {}, []
for t, neurons in selected.items():
    hyps = sorted({neuron2hyp[n] for n in neurons if n in neuron2hyp})
    if len(hyps) < 2:
        continue
    m, df, dropped = ge.score_hypotheses_stable({h: llm[h] for h in hyps},
                                                y_true=held_targets[t].astype(float))
    dropped_all += [dict(d, target=t) for d in dropped]
    if m is None:
        stable[t] = None
        continue
    stable[t] = {'auroc': m.get('auroc'), 'r2': m.get('r2'),
                 'significant': m['Significant'][0], 'n_hyp': m['Significant'][1],
                 'n_dropped': len(dropped)}
ge.log_json('12_evaluation_stable', {'per_target': stable, 'dropped': dropped_all})
print('dropped (zero-variance / duplicate):', len(dropped_all))
pd.DataFrame(stable).T

## Cosine Annotation

In [ ]:
te = get_local_embeddings(held_texts, model=ge.EMBEDDER, nomic_prefix='search_document: ',
                          cache_name='followup_held_doc', batch_size=128)
he = get_local_embeddings(union_hyps, model=ge.EMBEDDER, nomic_prefix='search_query: ',
                          cache_name='followup_hyp_query', batch_size=64)
T = np.stack([te[t] for t in held_texts]).astype(np.float32)
H = np.stack([he[h] for h in union_hyps]).astype(np.float32)
T /= np.linalg.norm(T, axis=1, keepdims=True)
H /= np.linalg.norm(H, axis=1, keepdims=True)
COS = T @ H.T
del te, he, T, H; ge.clear_mem()
COS.shape

## Cosine vs LLM Agreement

In [ ]:
agree = {}
for j, h in enumerate(union_hyps):
    a = llm[h].astype(float)
    if a.std() > 0:
        agree[h] = float(pearsonr(COS[:, j], a)[0])
mean_agreement = float(np.mean(list(agree.values()))) if agree else None
ge.log_json('09_cosine_annotation', {'mean_pearson_cosine_vs_llm': mean_agreement,
                                     'n_compared': len(agree), 'per_hypothesis': agree})
print('hypotheses compared:', len(agree))
print('mean pearson (cosine vs LLM):', mean_agreement)

## Predictiveness Comparison

In [ ]:
def separation(a, y):
    p, n = a == 1, a == 0
    return float(y[p].mean() - y[n].mean()) if p.sum() and n.sum() else np.nan

rows = []
for t, neurons in selected.items():
    y = held_targets[t].astype(float)
    for n in neurons:
        h = neuron2hyp.get(n)
        if not h:
            continue
        j = union_hyps.index(h)
        cos_bin = (COS[:, j] >= np.quantile(COS[:, j], 0.9)).astype(int)
        rows.append({'target': t, 'neuron': n, 'hypothesis': h,
                     'llm_sep': separation(llm[h], y),
                     'cos_sep': separation(cos_bin, y)})
cmp_df = pd.DataFrame(rows).dropna(subset=['llm_sep', 'cos_sep'])
rho = float(spearmanr(cmp_df['llm_sep'], cmp_df['cos_sep']).correlation)
ge.log_json('09_cosine_annotation', {'spearman_predictiveness_llm_vs_cosine': rho,
                                     'n_pairs': len(cmp_df),
                                     'comparison': cmp_df.to_dict('records')})
print('spearman(llm_sep, cos_sep) =', round(rho, 3), 'over', len(cmp_df), 'hypotheses')
cmp_df.head(10)

## Annotator Swap

In [ ]:
from google.colab import userdata
try:
    tok = userdata.get('HF_TOKEN')
    if tok:
        os.environ['HF_TOKEN'] = tok
except Exception:
    pass

# unsloth mirror is the same weights, ungated (meta-llama/* needs an approved token)
SWAP_MODELS = ['Qwen/Qwen3-0.6B', 'unsloth/Llama-3.2-1B-Instruct']

for m in SWAP_MODELS:
    n = ge.annotation_cache_missing('swap_' + m.split('/')[-1], union_hyps, held_texts)
    print(f'{m}: {n} uncached annotations' + ('  (cached, will skip serving)' if n == 0 else ''))
ge.kill_stray_vllm()
print('free VRAM (GB):', ge.gpu_free_gb())

In [ ]:
swap, swap_dropped = {}, {}
for model in SWAP_MODELS:
    cname = 'swap_' + model.split('/')[-1]
    missing = ge.annotation_cache_missing(cname, union_hyps, held_texts)
    proc = None
    try:
        if missing:
            proc = ge.serve_vllm(model, port=8001, max_model_len=2048,
                                 gpu_frac=0.60, enforce_eager=True)
        kw = {'temperature': 0.0, 'max_output_tokens': 64}
        if 'Qwen3' in model:   # Llama has no thinking mode; disable Qwen's for a fair comparison
            kw['extra_body'] = {'chat_template_kwargs': {'enable_thinking': False}}
        ann_m = annotate_texts_with_concepts(texts=held_texts, concepts=union_hyps, model=model,
                                             cache_name=cname, n_workers=16,
                                             max_words_per_example=64, **kw)
        sig, drops = {}, []
        for t, neurons in selected.items():
            hyps = sorted({neuron2hyp[n] for n in neurons if n in neuron2hyp})
            if len(hyps) < 2:
                continue
            m, df, dropped = ge.score_hypotheses_stable({h: ann_m[h] for h in hyps},
                                                        y_true=held_targets[t].astype(float))
            drops += [dict(d, target=t) for d in dropped]
            if m is None:
                continue
            sig[t] = {'significant': m['Significant'][0], 'n_hyp': m['Significant'][1],
                      'auroc': m.get('auroc')}
        swap[model] = sig
        swap_dropped[model] = drops
        print(model, '->', {t: v['significant'] for t, v in sig.items()})
    finally:
        ge.stop_vllm(proc)

ge.log_json('10_annotator_swap', {'significant_by_model': swap,
                                  'dropped_by_model': swap_dropped})
pd.DataFrame({m: {t: v['significant'] for t, v in s.items()} for m, s in swap.items()})

## Swap Stability

In [ ]:
base = {t: v['significant'] for t, v in
        {k: v for k, v in swap.get('Qwen/Qwen3-0.6B', {}).items()}.items()}
tbl = pd.DataFrame({m.split('/')[-1]: {t: v['significant'] for t, v in s.items()}
                    for m, s in swap.items()})
tbl['agree'] = tbl.nunique(axis=1) == 1
ge.log_json('10_annotator_swap', {'stability_table': tbl.to_dict()})
tbl